In [31]:
import csv, re, unicodedata
from collections import defaultdict

CSV_PATH = 'routes.csv'

In [32]:
SV_RAW = """abo turku|alavo alavus|birkala pirkkala|bjorneborg pori|borga porvoo|brahestad raahe|esbo espoo|fredrikshamn hamina|grankulla kauniainen|hango hanko|helsingfors helsinki|hogfors karkkila|hyvinge hyvinkaa|idensalmi iisalmi|ikalis ikaalinen|ilmola ilmajoki|inga inkoo|jakobstad pietarsaari|kajana kajaani|karleby kokkola|kasko kaskinen|kervo kerava|keuru keuruu|kimitoon kemionsaari|korsholm mustasaari|kristinestad kristiinankaupunki|kronoby kruunupyy|kumo kokemaki|kyrkslatt kirkkonummi|lahtis lahti|lappo lapua|larsmo luoto|lojo lohja|lovisa loviisa|lundo lieto|malax maalahti|nadendal naantali|narpes narpio|nykarleby uusikaarlepyy|nystad uusikaupunki|pargas parainen|raseborg raasepori|raumo rauma|reso raisio|s:t karins kaarina|s:t michel mikkeli|sibbo sipoo|sjundea siuntio|tammerfors tampere|tavastehus hameenlinna|tornea tornio|traskanda jarvenpaa|tusby tuusula|uleaborg oulu|ulvsby ulvila|vanda vantaa|vasa vaasa|vichtis vihti|villmanstrand lappeenranta|virmo mynamaki|vora voyri"""
SV = dict(p.rsplit(" ", 1) for p in SV_RAW.split("|"))
len(SV)

61

In [33]:
def norm(s):
    s = s.strip().strip(".").strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s).strip(" .")
    return SV.get(s, s)

norm(" Helsingfors "), norm("Hangö"), norm("ORIVESI"), norm("Imatra.")

('helsinki', 'hanko', 'orivesi', 'imatra')

In [34]:
edges, towns = set(), set()
with open(CSV_PATH, encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        a, b = norm(r["from_town"]), norm(r["to_town"])
        towns |= {a, b}
        if a != b:
            edges.add((min(a, b), max(a, b)))
print(len(towns), "towns,", len(edges), "edges")

116 towns, 448 edges


In [35]:
deg = defaultdict(int)
for a, b in edges:
    deg[a] += 1
    deg[b] += 1
rank = sorted(deg.items(), key=lambda x: (-x[1], x[0]))
for t, d in rank[:10]:
    print(f"{t:<15}{d}")

helsinki       39
orivesi        38
tornio         33
lieksa         29
imatra         27
nokia          23
eura           22
narpio         21
kemionsaari    19
inari          14


In [36]:
assert len(towns) == 116, len(towns)
assert len(edges) == 448, len(edges)
assert len({d for _, d in rank[:8]}) == 8, "tie in top 8"
print("All checks passed")

All checks passed


In [37]:
flag = "SIGMOID_" + "".join(t[0] for t, _ in rank[:7]).upper()
print(flag)

SIGMOID_HOTLINE
